# Task 1: MRV for 4-Queens

In [1]:
def is_safe(assignment, var, value):
    """Check if placing queen at (var, value) is safe"""
    # var is column index (0-3), value is row (1-4)
    for col, row in assignment.items():
        # Check same row
        if row == value:
            return False
        # Check diagonal
        if abs(col - var) == abs(row - value):
            return False
    return True

def select_mrv_variable(domains, assignment):
    """Return variable with smallest domain size"""
    unassigned = [v for v in domains if v not in assignment]
    return min(unassigned, key=lambda var: len(domains[var]))

def backtrack(assignment, domains):
    """Backtracking with MRV"""
    if len(assignment) == len(domains):
        return assignment
    
    var = select_mrv_variable(domains, assignment)
    print(f"Selecting variable: Column {var+1} (domain: {domains[var]})")
    
    for value in domains[var]:
        if is_safe(assignment, var, value):
            # Assign value
            assignment[var] = value
            print(f"  Trying Column {var+1} = Row {value}")
            
            # Update domains (remove assigned row from other columns)
            new_domains = {v: [r for r in domains[v] if r != value] for v in domains}
            new_domains[var] = [value]
            
            result = backtrack(assignment, new_domains)
            if result:
                return result
            
            # Backtrack
            del assignment[var]
            print(f"  Backtracking from Column {var+1} = Row {value}")
    
    return None

# Driver
domains = {i: [1, 2, 3, 4] for i in range(4)}
print("=== 4-Queens Problem with MRV ===\n")
solution = backtrack({}, domains)

print("\n=== Solution ===")
if solution:
    for col in range(4):
        print(f"Column {col+1} → Row {solution[col]}")
    
    # Visual representation
    print("\nBoard:")
    for row in range(1, 5):
        line = ""
        for col in range(4):
            if solution[col] == row:
                line += " Q "
            else:
                line += " . "
        print(line)

=== 4-Queens Problem with MRV ===

Selecting variable: Column 1 (domain: [1, 2, 3, 4])
  Trying Column 1 = Row 1
Selecting variable: Column 2 (domain: [2, 3, 4])
  Trying Column 2 = Row 3
Selecting variable: Column 3 (domain: [2, 4])
  Backtracking from Column 2 = Row 3
  Trying Column 2 = Row 4
Selecting variable: Column 3 (domain: [2, 3])
  Trying Column 3 = Row 2
Selecting variable: Column 4 (domain: [3])
  Backtracking from Column 3 = Row 2
  Backtracking from Column 2 = Row 4
  Backtracking from Column 1 = Row 1
  Trying Column 1 = Row 2
Selecting variable: Column 2 (domain: [1, 3, 4])
  Trying Column 2 = Row 4
Selecting variable: Column 3 (domain: [1, 3])
  Trying Column 3 = Row 1
Selecting variable: Column 4 (domain: [3])
  Trying Column 4 = Row 3

=== Solution ===
Column 1 → Row 2
Column 2 → Row 4
Column 3 → Row 1
Column 4 → Row 3

Board:
 .  .  Q  . 
 Q  .  .  . 
 .  .  .  Q 
 .  Q  .  . 


# Task 2: LCV Map Coloring

In [2]:
def lcv_order(var, domains, neighbors):
    """Sort values based on least constraint (fewest eliminations)"""
    def count_eliminations(value):
        """Count how many neighbor values would be eliminated"""
        count = 0
        for neighbor in neighbors[var]:
            if value in domains[neighbor]:
                count += 1
        return count
    
    return sorted(domains[var], key=count_eliminations)

def backtrack(assignment, domains, neighbors, variables):
    """Simple backtracking with LCV"""
    if len(assignment) == len(variables):
        return assignment
    
    # Select first unassigned variable (no MRV for this task)
    var = next(v for v in variables if v not in assignment)
    
    # Get LCV ordered values
    ordered_values = lcv_order(var, domains, neighbors)
    print(f"Variable Selected: {var}")
    print(f"LCV Order: {ordered_values}")
    
    for value in ordered_values:
        # Check consistency with neighbors
        consistent = True
        for neighbor in neighbors[var]:
            if neighbor in assignment and assignment[neighbor] == value:
                consistent = False
                break
        
        if consistent:
            assignment[var] = value
            result = backtrack(assignment, domains, neighbors, variables)
            if result:
                return result
            del assignment[var]
    
    return None

# Problem definition
variables = ['WA', 'NT', 'SA', 'Q', 'NSW', 'V']
domains = {v: ['Red', 'Green', 'Blue'] for v in variables}

neighbors = {
    'WA': ['NT', 'SA'],
    'NT': ['WA', 'SA', 'Q'],
    'SA': ['WA', 'NT', 'Q', 'NSW', 'V'],
    'Q': ['NT', 'SA', 'NSW'],
    'NSW': ['SA', 'Q', 'V'],
    'V': ['SA', 'NSW']
}

print("\n=== Map Coloring with LCV ===\n")
solution = backtrack({}, domains, neighbors, variables)

print("\n=== Final valid assignment ===")
if solution:
    output = ", ".join([f"{v} = {solution[v]}" for v in variables])
    print(output)


=== Map Coloring with LCV ===

Variable Selected: WA
LCV Order: ['Red', 'Green', 'Blue']
Variable Selected: NT
LCV Order: ['Red', 'Green', 'Blue']
Variable Selected: SA
LCV Order: ['Red', 'Green', 'Blue']
Variable Selected: Q
LCV Order: ['Red', 'Green', 'Blue']
Variable Selected: NSW
LCV Order: ['Red', 'Green', 'Blue']
Variable Selected: V
LCV Order: ['Red', 'Green', 'Blue']

=== Final valid assignment ===
WA = Red, NT = Green, SA = Blue, Q = Red, NSW = Green, V = Red


# Task 3: AC-3

In [3]:
def revise(Xi, Xj, domains, constraints):
    """Remove values from domain of Xi that have no support in Xj"""
    revised = False
    
    for x in domains[Xi][:]:  # Iterate over copy
        # Check if there exists y in Xj such that constraint holds
        has_support = False
        for y in domains[Xj]:
            if constraints.get((Xi, Xj), lambda a, b: True)(x, y):
                has_support = True
                break
        
        if not has_support:
            domains[Xi].remove(x)
            revised = True
            print(f"Revise({Xi}, {Xj}): Removed {x}")
    
    return revised

def ac3(variables, domains, constraints):
    """AC-3 algorithm for arc consistency"""
    from collections import deque
    
    # Initialize queue with all arcs
    queue = deque()
    for (xi, xj) in constraints.keys():
        queue.append((xi, xj))
    
    while queue:
        (xi, xj) = queue.popleft()
        
        if revise(xi, xj, domains, constraints):
            if not domains[xi]:
                return False
            
            # Add all arcs (xk, xi) where xk != xj
            for (xk, xj2) in constraints.keys():
                if xk == xi and xj2 != xj:
                    queue.append((xk, xj2))
    
    return True

# Problem definition
variables = ['X1', 'X2', 'X3']
domains = {v: [1, 2, 3, 4] for v in variables}

# Define constraints as lambda functions
def less_than(a, b):
    return a < b

constraints = {
    ('X1', 'X2'): less_than,
    ('X2', 'X1'): lambda a, b: b < a,  # Inverse constraint
    ('X2', 'X3'): less_than,
    ('X3', 'X2'): lambda a, b: b < a,
}

print("\n=== AC-3 Algorithm ===\n")
print("Initial domains:", domains)
print("\nProcessing arcs:")

ac3(variables, domains, constraints)

print("\n=== Final domains ===")
for v in variables:
    print(f"D({v}) = {domains[v]}")


=== AC-3 Algorithm ===

Initial domains: {'X1': [1, 2, 3, 4], 'X2': [1, 2, 3, 4], 'X3': [1, 2, 3, 4]}

Processing arcs:
Revise(X1, X2): Removed 4
Revise(X2, X1): Removed 1
Revise(X2, X3): Removed 4
Revise(X3, X2): Removed 1
Revise(X3, X2): Removed 2

=== Final domains ===
D(X1) = [1, 2, 3]
D(X2) = [2, 3]
D(X3) = [3, 4]


# Task 4: Sudoku AC-3

In [4]:
def get_neighbors(cell, grid_size=4):
    """Get all cells that share row, column, or subgrid with given cell"""
    r, c = cell
    neighbors = set()
    
    # Same row
    for col in range(grid_size):
        if col != c:
            neighbors.add((r, col))
    
    # Same column
    for row in range(grid_size):
        if row != r:
            neighbors.add((row, c))
    
    # Same 2x2 subgrid
    sub_r, sub_c = r // 2, c // 2
    for i in range(2):
        for j in range(2):
            nr, nc = sub_r * 2 + i, sub_c * 2 + j
            if (nr, nc) != (r, c):
                neighbors.add((nr, nc))
    
    return neighbors

def revise_sudoku(cell1, cell2, domains):
    """Revise for Sudoku - remove values that conflict with cell2"""
    revised = False
    
    if cell1 not in domains or cell2 not in domains:
        return False
    
    for val in domains[cell1][:]:
        # If cell2 has only this value and it's the same, remove from cell1
        if len(domains[cell2]) == 1 and domains[cell2][0] == val:
            domains[cell1].remove(val)
            revised = True
            print(f"Revise({cell1}, {cell2}): Removed {val}")
    
    return revised

def ac3_sudoku(grid):
    """Apply AC-3 to Sudoku grid"""
    from collections import deque
    
    # Variables: empty cells (where grid[row][col] == 0)
    grid_size = len(grid)
    empty_cells = [(r, c) for r in range(grid_size) for c in range(grid_size) if grid[r][c] == 0]
    
    # Domains: possible values for each empty cell
    domains = {}
    for r, c in empty_cells:
        used = set()
        # Check row
        for col in range(grid_size):
            if grid[r][col] != 0:
                used.add(grid[r][col])
        # Check column
        for row in range(grid_size):
            if grid[row][c] != 0:
                used.add(grid[row][c])
        # Check 2x2 subgrid
        sr, sc = r // 2, c // 2
        for i in range(2):
            for j in range(2):
                val = grid[sr*2 + i][sc*2 + j]
                if val != 0:
                    used.add(val)
        
        domains[(r, c)] = [v for v in range(1, 5) if v not in used]
    
    # Initialize queue with all arcs
    queue = deque()
    for cell in empty_cells:
        for neighbor in get_neighbors(cell):
            if neighbor in domains:
                queue.append((cell, neighbor))
    
    # AC-3 algorithm
    while queue:
        cell1, cell2 = queue.popleft()
        
        if revise_sudoku(cell1, cell2, domains):
            if not domains[cell1]:
                return domains
            # Add arcs (cell3, cell1) for all cell3
            for neighbor in get_neighbors(cell1):
                if neighbor in domains and neighbor != cell2:
                    queue.append((neighbor, cell1))
    
    return domains

# Sudoku grid
grid = [
    [1, 0, 0, 4],
    [0, 0, 3, 0],
    [0, 3, 0, 0],
    [2, 0, 0, 1]
]

print("\n=== Sudoku AC-3 (4x4) ===\n")
print("Initial Grid:")
for row in grid:
    print(row)

print("\nProcessing arcs...\n")
reduced_domains = ac3_sudoku(grid)

print("\n=== Reduced Domains ===")
for cell in sorted(reduced_domains.keys()):
    print(f"Cell {cell} → {reduced_domains[cell]}")


=== Sudoku AC-3 (4x4) ===

Initial Grid:
[1, 0, 0, 4]
[0, 0, 3, 0]
[0, 3, 0, 0]
[2, 0, 0, 1]

Processing arcs...

Revise((0, 1), (0, 2)): Removed 2

=== Reduced Domains ===
Cell (0, 1) → []
Cell (0, 2) → [2]
Cell (1, 0) → [4]
Cell (1, 1) → [2, 4]
Cell (1, 3) → [2]
Cell (2, 0) → [4]
Cell (2, 2) → [2, 4]
Cell (2, 3) → [2]
Cell (3, 1) → [4]
Cell (3, 2) → [4]


# Task 5: Cryptarithmetic Modeling

In [5]:
# TASK 5: Cryptarithmetic CSP Modeling
# SEND + MORE = MONEY

# -------------------------------
# 1. VARIABLES
# -------------------------------
letters = ['S', 'E', 'N', 'D', 'M', 'O', 'R', 'Y']
carries = ['C1', 'C2', 'C3', 'C4']

variables = letters + carries

# -------------------------------
# 2. DOMAINS
# -------------------------------
domains = {}

# Assign domain {0–9} to each letter
for letter in letters:
    domains[letter] = list(range(10))

# Assign domain {0,1} to carry variables
for carry in carries:
    domains[carry] = [0, 1]

# Apply leading digit constraint (S ≠ 0, M ≠ 0)
domains['S'] = [d for d in domains['S'] if d != 0]
domains['M'] = [d for d in domains['M'] if d != 0]

# -------------------------------
# 3. CONSTRAINTS
# -------------------------------

# ALL-DIFFERENT constraint for letters (expressed as a function)
def all_different(assignment):
    values = [assignment[l] for l in letters if l in assignment]
    return len(values) == len(set(values))

# Column-wise arithmetic constraints:
# D + E = Y + 10*C1
# N + R + C1 = E + 10*C2
# E + O + C2 = N + 10*C3
# S + M + C3 = O + 10*C4
# C4 = M

def check_constraints(assignment):
    """Check all constraints"""
    # Need all variables assigned
    required = set(letters + carries)
    if not required.issubset(set(assignment.keys())):
        return True  # Not all assigned yet
    
    # All different constraint
    if len(set(assignment[l] for l in letters)) != len(letters):
        return False
    
    # Column constraints
    if assignment['D'] + assignment['E'] != assignment['Y'] + 10 * assignment['C1']:
        return False
    if assignment['N'] + assignment['R'] + assignment['C1'] != assignment['E'] + 10 * assignment['C2']:
        return False
    if assignment['E'] + assignment['O'] + assignment['C2'] != assignment['N'] + 10 * assignment['C3']:
        return False
    if assignment['S'] + assignment['M'] + assignment['C3'] != assignment['O'] + 10 * assignment['C4']:
        return False
    if assignment['C4'] != assignment['M']:
        return False
    
    return True

# -------------------------------
# 4. OUTPUT
# -------------------------------

print("\n=== Cryptarithmetic CSP Modeling: SEND + MORE = MONEY ===\n")

print("Variables:", variables)

print("\nDomains:")
for v in variables:
    print(f"{v} → {domains.get(v, 'Not Assigned')[:5]}..." if len(domains.get(v, [])) > 5 else f"{v} → {domains.get(v, 'Not Assigned')}")

print("\nConstraints:")
print("1. All-different constraint (S,E,N,D,M,O,R,Y all different)")
print("2. Leading digits cannot be zero (S ≠ 0, M ≠ 0)")
print("3. Column-wise arithmetic constraints:")
print("   D + E = Y + 10*C1")
print("   N + R + C1 = E + 10*C2")
print("   E + O + C2 = N + 10*C3")
print("   S + M + C3 = O + 10*C4")
print("   C4 = M")
print("\nNote: C1, C2, C3, C4 are carry variables (0 or 1)")


=== Cryptarithmetic CSP Modeling: SEND + MORE = MONEY ===

Variables: ['S', 'E', 'N', 'D', 'M', 'O', 'R', 'Y', 'C1', 'C2', 'C3', 'C4']

Domains:
S → [1, 2, 3, 4, 5]...
E → [0, 1, 2, 3, 4]...
N → [0, 1, 2, 3, 4]...
D → [0, 1, 2, 3, 4]...
M → [1, 2, 3, 4, 5]...
O → [0, 1, 2, 3, 4]...
R → [0, 1, 2, 3, 4]...
Y → [0, 1, 2, 3, 4]...
C1 → [0, 1]
C2 → [0, 1]
C3 → [0, 1]
C4 → [0, 1]

Constraints:
1. All-different constraint (S,E,N,D,M,O,R,Y all different)
2. Leading digits cannot be zero (S ≠ 0, M ≠ 0)
3. Column-wise arithmetic constraints:
   D + E = Y + 10*C1
   N + R + C1 = E + 10*C2
   E + O + C2 = N + 10*C3
   S + M + C3 = O + 10*C4
   C4 = M

Note: C1, C2, C3, C4 are carry variables (0 or 1)


# Task 6: TWO + TWO = FOUR

In [6]:
def select_mrv_variable(domains, assignment):
    """MRV heuristic - choose variable with smallest domain"""
    unassigned = [v for v in domains if v not in assignment]
    return min(unassigned, key=lambda var: len(domains[var]))

def lcv_order(var, domains, assignment, neighbors):
    """LCV heuristic - order values by fewest conflicts"""
    def count_conflicts(value):
        count = 0
        for neighbor in neighbors.get(var, []):
            if neighbor in assignment and assignment[neighbor] == value:
                count += 1
        return count
    
    values = domains[var]
    return sorted(values, key=count_conflicts)

def is_consistent(var, value, assignment, constraints):
    """Check if assignment is consistent with constraints"""
    temp_assignment = assignment.copy()
    temp_assignment[var] = value
    
    # Apply all constraints
    for constraint in constraints:
        if not constraint(temp_assignment):
            return False
    return True

def backtrack(assignment, domains, constraints, neighbors):
    """Backtracking with MRV and LCV"""
    if len(assignment) == len(domains):
        return assignment
    
    var = select_mrv_variable(domains, assignment)
    
    # Get LCV ordered values
    ordered_values = lcv_order(var, domains, assignment, neighbors)
    
    for value in ordered_values:
        if is_consistent(var, value, assignment, constraints):
            assignment[var] = value
            result = backtrack(assignment, domains, constraints, neighbors)
            if result:
                return result
            del assignment[var]
    
    return None

# Problem: TWO + TWO = FOUR
# Variables: T, W, O, F, U, R
# Carries: C1, C2, C3

letters = ['T', 'W', 'O', 'F', 'U', 'R']
carries = ['C1', 'C2', 'C3']
variables = letters + carries

# Domains
domains = {v: list(range(10)) for v in letters}
for c in carries:
    domains[c] = [0, 1]

# Leading digit constraints
domains['T'] = [d for d in domains['T'] if d != 0]
domains['F'] = [d for d in domains['F'] if d != 0]

# Neighbors for LCV (variables that interact)
neighbors = {
    'T': ['W', 'O', 'F', 'U', 'R', 'C3'],
    'W': ['T', 'O', 'F', 'U', 'R', 'C2'],
    'O': ['T', 'W', 'F', 'U', 'R', 'C1', 'C2', 'C3'],
    'F': ['T', 'W', 'O', 'U', 'R', 'C3'],
    'U': ['T', 'W', 'O', 'F', 'R', 'C2'],
    'R': ['T', 'W', 'O', 'F', 'U', 'C1'],
    'C1': ['O', 'R', 'W'],
    'C2': ['O', 'U', 'W'],
    'C3': ['O', 'T', 'F']
}

# Constraints
def constraint1(assign):
    """O + O = R + 10*C1"""
    if all(v in assign for v in ['O', 'R', 'C1']):
        return (assign['O'] + assign['O']) == (assign['R'] + 10 * assign['C1'])
    return True

def constraint2(assign):
    """W + W + C1 = U + 10*C2"""
    if all(v in assign for v in ['W', 'C1', 'U', 'C2']):
        return (assign['W'] + assign['W'] + assign['C1']) == (assign['U'] + 10 * assign['C2'])
    return True

def constraint3(assign):
    """T + T + C2 = O + 10*C3"""
    if all(v in assign for v in ['T', 'C2', 'O', 'C3']):
        return (assign['T'] + assign['T'] + assign['C2']) == (assign['O'] + 10 * assign['C3'])
    return True

def constraint4(assign):
    """C3 = F"""
    if all(v in assign for v in ['C3', 'F']):
        return assign['C3'] == assign['F']
    return True

def all_different(assign):
    """All letters must have different values"""
    letter_values = [assign[l] for l in letters if l in assign]
    return len(letter_values) == len(set(letter_values))

constraints = [constraint1, constraint2, constraint3, constraint4, all_different]

print("\n=== TWO + TWO = FOUR Solver (MRV + LCV) ===\n")

solution = backtrack({}, domains, constraints, neighbors)

print("\n=== Solution ===")
if solution:
    print(f"T = {solution['T']}")
    print(f"W = {solution['W']}")
    print(f"O = {solution['O']}")
    print(f"F = {solution['F']}")
    print(f"U = {solution['U']}")
    print(f"R = {solution['R']}")
    print(f"\nVerification: {solution['T']}{solution['W']}{solution['O']} + {solution['T']}{solution['W']}{solution['O']} = {solution['F']}{solution['O']}{solution['U']}{solution['R']}")
    print(f"  {solution['T']}{solution['W']}{solution['O']} + {solution['T']}{solution['W']}{solution['O']} = {2 * (100*solution['T'] + 10*solution['W'] + solution['O'])}")


=== TWO + TWO = FOUR Solver (MRV + LCV) ===


=== Solution ===
T = 7
W = 3
O = 4
F = 1
U = 6
R = 8

Verification: 734 + 734 = 1468
  734 + 734 = 1468


# Task 7: Exam Timetabling

In [7]:
# Exams
exams = ['E1', 'E2', 'E3', 'E4', 'E5']
slots = ['Morning', 'Afternoon', 'Evening']

# Domains
domains = {exam: slots.copy() for exam in exams}

# Conflicts: exams that cannot be scheduled together (share students)
conflicts = {
    'E1': ['E2', 'E3'],
    'E2': ['E1', 'E3', 'E4'],
    'E3': ['E1', 'E2', 'E5'],
    'E4': ['E2', 'E5'],
    'E5': ['E3', 'E4']
}

# Capacity constraint: max 2 exams per slot
capacity = 2

# Administrative constraints: some exams must be earlier
# E1 must be before E4, E2 before E5
slot_order = {'Morning': 0, 'Afternoon': 1, 'Evening': 2}

def select_mrv(domains, assignment):
    """MRV: select exam with smallest domain"""
    unassigned = [e for e in exams if e not in assignment]
    return min(unassigned, key=lambda e: len(domains[e]))

def lcv(exam, domains, assignment, conflicts, capacity):
    """LCV: order slots by least constraint"""
    def count_conflicts(slot):
        conflicts_count = 0
        # Count exams in same slot
        for other, assigned_slot in assignment.items():
            if assigned_slot == slot:
                conflicts_count += 1
        # Count constraints violated
        for conflicting in conflicts.get(exam, []):
            if conflicting in assignment and assignment[conflicting] == slot:
                conflicts_count += 10  # High penalty
        return conflicts_count
    
    slots_ordered = sorted(domains[exam], key=count_conflicts)
    print(f"  LCV order for {exam}: {slots_ordered}")
    return slots_ordered

def is_consistent(exam, slot, assignment, conflicts, capacity):
    """Check if assignment is consistent"""
    # Check capacity
    count_in_slot = sum(1 for s in assignment.values() if s == slot)
    if count_in_slot >= capacity:
        return False
    
    # Check conflicts
    for conflicting in conflicts.get(exam, []):
        if conflicting in assignment and assignment[conflicting] == slot:
            return False
    
    return True

def backtrack(assignment, domains, conflicts, capacity):
    """Backtracking with MRV and LCV"""
    if len(assignment) == len(exams):
        return assignment
    
    exam = select_mrv(domains, assignment)
    print(f"\nSelecting exam: {exam} (domain: {domains[exam]})")
    
    # Get LCV ordered slots
    ordered_slots = lcv(exam, domains, assignment, conflicts, capacity)
    
    for slot in ordered_slots:
        if is_consistent(exam, slot, assignment, conflicts, capacity):
            assignment[exam] = slot
            print(f"  Assigning {exam} → {slot}")
            
            # Update domains (remove slot if capacity reached)
            new_domains = {e: d.copy() for e, d in domains.items()}
            count_in_slot = sum(1 for s in assignment.values() if s == slot)
            if count_in_slot >= capacity:
                for e in exams:
                    if e not in assignment and slot in new_domains[e]:
                        new_domains[e].remove(slot)
            
            result = backtrack(assignment, new_domains, conflicts, capacity)
            if result:
                return result
            
            del assignment[exam]
            print(f"  Backtracking from {exam} → {slot}")
    
    return None

print("\n=== Exam Timetabling (MRV + LCV) ===\n")

solution = backtrack({}, domains, conflicts, capacity)

print("\n=== Final Exam Schedule ===")
if solution:
    for exam in exams:
        print(f"{exam} → {solution[exam]}")
    
    print("\nSchedule by slot:")
    for slot in slots:
        exams_in_slot = [e for e in exams if solution[e] == slot]
        print(f"{slot}: {exams_in_slot if exams_in_slot else 'None'}")


=== Exam Timetabling (MRV + LCV) ===


Selecting exam: E1 (domain: ['Morning', 'Afternoon', 'Evening'])
  LCV order for E1: ['Morning', 'Afternoon', 'Evening']
  Assigning E1 → Morning

Selecting exam: E2 (domain: ['Morning', 'Afternoon', 'Evening'])
  LCV order for E2: ['Afternoon', 'Evening', 'Morning']
  Assigning E2 → Afternoon

Selecting exam: E3 (domain: ['Morning', 'Afternoon', 'Evening'])
  LCV order for E3: ['Evening', 'Morning', 'Afternoon']
  Assigning E3 → Evening

Selecting exam: E4 (domain: ['Morning', 'Afternoon', 'Evening'])
  LCV order for E4: ['Morning', 'Evening', 'Afternoon']
  Assigning E4 → Morning

Selecting exam: E5 (domain: ['Afternoon', 'Evening'])
  LCV order for E5: ['Afternoon', 'Evening']
  Assigning E5 → Afternoon

=== Final Exam Schedule ===
E1 → Morning
E2 → Afternoon
E3 → Evening
E4 → Morning
E5 → Afternoon

Schedule by slot:
Morning: ['E1', 'E4']
Afternoon: ['E2', 'E5']
Evening: ['E3']


# Task 8: Frequency Assignment (AC-3)

In [8]:
from collections import deque

# Towers
towers = ['T1', 'T2', 'T3', 'T4']
frequencies = ['F1', 'F2', 'F3', 'F4']

# Domains
domains = {
    'T1': ['F1', 'F2', 'F3', 'F4'],
    'T2': ['F1', 'F2', 'F3', 'F4'],
    'T3': ['F1', 'F2', 'F3', 'F4'],
    'T4': ['F1', 'F2', 'F3', 'F4']
}

# Hardware limitations (certain towers cannot use specific frequencies)
# T1 cannot use F1
domains['T1'] = [f for f in domains['T1'] if f != 'F1']
# T3 cannot use F4
domains['T3'] = [f for f in domains['T3'] if f != 'F4']

print("=== Frequency Assignment (AC-3) ===\n")
print("Initial domains:")
for t in towers:
    print(f"{t} → {domains[t]}")
print()

# Neighbors (adjacent towers cannot use same frequency)
neighbors = {
    'T1': ['T2', 'T3'],
    'T2': ['T1', 'T4'],
    'T3': ['T1', 'T4'],
    'T4': ['T2', 'T3']
}

def revise(xi, xj, domains, neighbors):
    """Remove values from domain of xi that have no support in xj"""
    revised = False
    
    for val in domains[xi][:]:
        # Check if there exists a value in xj that is different (not equal)
        has_support = False
        for other_val in domains[xj]:
            if val != other_val:
                has_support = True
                break
        
        if not has_support:
            domains[xi].remove(val)
            revised = True
            print(f"Arc ({xi}, {xj}): Removed {val} from {xi}")
    
    return revised

def ac3(domains, neighbors):
    """AC-3 algorithm for arc consistency"""
    # Initialize queue with all arcs
    queue = deque()
    for xi in neighbors:
        for xj in neighbors[xi]:
            queue.append((xi, xj))
    
    while queue:
        xi, xj = queue.popleft()
        
        if revise(xi, xj, domains, neighbors):
            if not domains[xi]:
                return False
            
            # Add all arcs (xk, xi) for xk in neighbors of xi
            for xk in neighbors.get(xi, []):
                if xk != xj:
                    queue.append((xk, xi))
    
    return True

# Run AC-3
print("Processing arcs...\n")
ac3(domains, neighbors)

print("\n=== Reduced Domains ===")
for t in towers:
    print(f"{t} → {domains[t]}")

=== Frequency Assignment (AC-3) ===

Initial domains:
T1 → ['F2', 'F3', 'F4']
T2 → ['F1', 'F2', 'F3', 'F4']
T3 → ['F1', 'F2', 'F3']
T4 → ['F1', 'F2', 'F3', 'F4']

Processing arcs...


=== Reduced Domains ===
T1 → ['F2', 'F3', 'F4']
T2 → ['F1', 'F2', 'F3', 'F4']
T3 → ['F1', 'F2', 'F3']
T4 → ['F1', 'F2', 'F3', 'F4']
